# Tutorial 5, Q1b: desert island knapsack

Release after the practical.

**The problem.** Table 1 of the sheet: eight items you could take to a desert island. You can carry 15 kg.
**Q1b:** as Q1a, but you may take an item more than once.

| Item | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Weight (kg) | 2 | 3 | 2 | 6 | 6 | 8 | 3 | 5 |
| Survival value | 10 | 50 | 40 | 100 | 100 | 130 | 70 | 80 |

$x_i$ is now the number of copies of item $i$: maximise $\sum_{i=1}^{8} s_i x_i$ subject to $\sum_{i=1}^{8} w_i x_i \le 15$, $x_i \ge 0$ and whole numbers (no upper limit of 1). In the code, items 1–8 are `x[0]` … `x[7]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Survival value **350**: five of item 7 (`x[6] = 5`), exactly 15 kg. The answer sheet shows the same. A value printed as `-0.0` is zero (a rounding effect in the solver).

*Original script: `5 Source Files/Python/knapsack_tutorial_b.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

The only change from Q1a: the variables are whole numbers $\ge 0$ (`NonNegativeIntegers`), not binary.

In [ ]:
# Define integer (not binary) variables: x[i] = how many of item i+1 you take, a whole number >= 0
model.x = Var(range(8), within=NonNegativeIntegers)

The data from Table 1.

In [ ]:
# Define the objective function values, the weights and the capacity
value = [10, 50, 40, 100, 100, 130, 70, 80]  # survival value of items 1-8
weight = [2, 3, 2, 6, 6, 8, 3, 5]               # weight of items 1-8 (kg)
W = 15                                          # you can carry 15 kg

The objective: total survival value, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(value[i] * model.x[i] for i in range(8)), sense=maximize)

The capacity: the items taken must not weigh more than 15 kg.

In [ ]:
# Define the capacity constraint: sum of w_i x_i <= W
model.capacity = Constraint(expr=sum(weight[i] * model.x[i] for i in range(8)) <= W)

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## Try it

Change `W = 15` to `W = 16` and run all cells. Is the answer still "as many of item 7 as fit"? Change it back
afterwards.

<details><summary>Answer</summary>

No. Five of item 7 still fit (15 kg, 350), but the optimum is **360**: four of item 7 and two of item 3
(12 + 4 = 16 kg). Picking the best value per kilogram first (a *greedy* rule) can leave capacity unused; the
integer model finds the best combination.
</details>